<a href="https://colab.research.google.com/github/mamoor2019/LLMBot/blob/main/NCERTlearningAsistant_mamoor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project Name**    - NCERT Learning Assistant: A Domain-Specific Large Language Model for School-Level Science and Mathematics Question Answering

**Project type:** Deep Learning for NLP and Applied Generative AI  
##### **Contribution -** Individual
##### **Submitted by -** Mohammad Mamoor Ahmad

## Project Summary

The objective of this project is to develop an AI-powered educational chatbot that helps school students understand concepts, solve problems, and clarify doubts based on NCERT textbooks. The system will use NCERT Science and Mathematics books as its primary knowledge source and leverage a pretrained Large Language Model (LLM) adapted for school-level educational question answering.

The initial project scope will cover Classes 6–10, with the possibility of extending it to Classes 11–12. The chatbot will provide student-friendly explanations, definitions, examples, revision notes, practice questions, and step-by-step mathematical solutions appropriate to the selected class.

The project will combine parameter-efficient fine-tuning using QLoRA with Retrieval-Augmented Generation (RAG). Fine-tuning will teach the model to respond in a clear, structured, and age-appropriate educational style, while RAG will retrieve relevant information from NCERT textbooks to ground answers in the prescribed curriculum.

**Project Objectives**
* Collect NCERT Science and Mathematics textbooks in PDF format.
* Extract, clean, and organize textbook content by class, subject, chapter, and page.
* Create a question-and-answer dataset for educational instruction and model fine-tuning.
* Fine-tune a pretrained Hugging Face LLM using QLoRA and parameter-efficient fine-tuning techniques.
* Implement a RAG pipeline using text embeddings and a vector database to retrieve relevant textbook passages.
* Develop an interactive chatbot that supports class- and subject-specific questions.
* Provide clear explanations, step-by-step mathematical solutions, and relevant textbook source references.
* Evaluate the system against baseline models to measure answer correctness, relevance, curriculum alignment, and educational usefulness.



# **GitHub Link -**

https://github.com/mamoor2019/LLMBot

# **Problem Statement**

Students often require additional explanations beyond those provided in textbooks. They may struggle to understand complex concepts, solve mathematical problems, or identify relevant information across chapters. General-purpose AI chatbots can provide useful explanations but may produce inaccurate information, answers beyond the student's grade level, or content that does not align with the NCERT curriculum.

This project addresses these challenges by developing a curriculum-focused chatbot that uses NCERT textbook content to provide relevant, understandable, and traceable answers.

The expected outcome is a working educational chatbot that demonstrates how fine-tuning and retrieval can be combined to support school-level learning. The project will also produce a prepared NCERT knowledge base, a fine-tuning dataset, an adapted language model, an interactive application, and a comparative evaluation of the proposed approach.

The research contribution will focus on determining whether combining domain-specific fine-tuning with NCERT-based retrieval improves the quality and curriculum alignment of educational question answering compared with a general-purpose LLM.

## 1. NCERT Data Collection

The input tree is `DataSet/ncert/class_6/` through `class_10/`, with subject folders such as Mathematics and Science beneath each class. Each PDF page is retained as a separate record with its source path, class, subject, one-based page number, and detected chapter. Missing grades are reported and skipped rather than treated as an error; add their PDFs under the same directory convention to include them.

### Environment and imports

Run the dependency-install cell once, then restart the Colab runtime before executing any imports or model-loading cells. After restart, continue from the imports cell in order. Avoid running another `pip install --upgrade` later in the notebook because that can leave already-imported Hugging Face modules out of sync.

In [1]:
%pip install -q "transformers>=4.45,<5" "huggingface_hub>=0.34,<1" "datasets>=4,<5" "peft>=0.13" "bitsandbytes>=0.44" accelerate sentence-transformers faiss-cpu gradio pypdf scikit-learn "pandas==2.2.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.8/407.8 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 80.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatibl

In [2]:
# Built-in utilities for paths, JSON, regular expressions, and reproducibility.
import json
import re
import random
from pathlib import Path

# DataFrame and array operations for page records, chunks, and evaluation.
import numpy as np
import pandas as pd
import torch

# Load textbook PDFs and create document-level dataset splits.
from datasets import Dataset
from pypdf import PdfReader
from sklearn.model_selection import GroupShuffleSplit

# Create semantic embeddings and search their FAISS vector index.
from sentence_transformers import SentenceTransformer
import faiss

# Load Phi-3 and configure quantized inference and supervised fine-tuning.
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

# Attach and train lightweight LoRA adapters on the quantized base model.
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Runtime device: {DEVICE}")

Runtime device: cuda


## 2. PDF Processing

PDFs are read page by page. The processor derives grade and subject from the directory structure, normalizes broken line wraps and whitespace, identifies explicit chapter headings when present, and keeps one-based page metadata for citations. Scanned PDFs without an OCR text layer are counted and reported so OCR can be added deliberately.

In [3]:
from google.colab import drive
from pathlib import Path

# Mount Google Drive to access the dataset
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
PROJECT_ROOT = Path.cwd()
OUTPUT_ROOT = PROJECT_ROOT / "artifacts"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Dynamic case-insensitive search in Google Drive
def find_dataset_path(base_path, target_parts):
    current = Path(base_path)
    for part in target_parts:
        matched = None
        if current.exists():
            for child in current.iterdir():
                if child.name.lower() == part.lower():
                    matched = child
                    break
        if matched is None:
            return None
        current = matched
    return current

# Search for "LLMBot/DataSEt" under Google Drive
resolved_path = find_dataset_path("/content/drive/MyDrive", ["LLMBot", "DataSEt"])

if resolved_path and resolved_path.exists():
    DATA_ROOT = resolved_path
    print(f"Data root successfully located: {DATA_ROOT.resolve()}")
    print(f"Output root: {OUTPUT_ROOT.resolve()}")
else:
    # Fallback to direct path definition if search fails
    DATA_ROOT = Path("/content/drive/MyDrive/LLMBot/DataSEt")
    if not DATA_ROOT.exists():
        raise FileNotFoundError(
            f"Could not automatically locate the dataset folder. Tried fallback: {DATA_ROOT}. "
            "Please verify the exact folder name spelling and capitalization inside your Google Drive."
        )
    else:
        print(f"Data root located via fallback: {DATA_ROOT.resolve()}")

Data root successfully located: /content/drive/MyDrive/LLMBot/DataSet
Output root: /content/artifacts


In [5]:
# Using the dynamically resolved DATA_ROOT from the previous cell
print(f"Data root: {DATA_ROOT.resolve()}")
print(f"Output root: {OUTPUT_ROOT.resolve()}")

Data root: /content/drive/MyDrive/LLMBot/DataSet
Output root: /content/artifacts


In [6]:
SUPPORTED_GRADES = {f"class_{grade}": grade for grade in range(6, 11)}
pdf_paths = sorted(
    path for path in DATA_ROOT.rglob("*")
    if path.is_file() and path.suffix.casefold() == ".pdf"
)

available_grades = sorted({
    part
    for path in pdf_paths
    for part in path.relative_to(DATA_ROOT).parts
    if part in SUPPORTED_GRADES
})
print(f"Found {len(pdf_paths)} PDF files")
print("Available grades:", available_grades)

Found 138 PDF files
Available grades: ['class_10', 'class_6', 'class_7', 'class_8', 'class_9']


In [7]:
pdf_paths[:10]

[PosixPath('/content/drive/MyDrive/LLMBot/DataSet/ncert/class_10/Mathematics/jemh101.pdf'),
 PosixPath('/content/drive/MyDrive/LLMBot/DataSet/ncert/class_10/Mathematics/jemh102.pdf'),
 PosixPath('/content/drive/MyDrive/LLMBot/DataSet/ncert/class_10/Mathematics/jemh103.pdf'),
 PosixPath('/content/drive/MyDrive/LLMBot/DataSet/ncert/class_10/Mathematics/jemh104.pdf'),
 PosixPath('/content/drive/MyDrive/LLMBot/DataSet/ncert/class_10/Mathematics/jemh105.pdf'),
 PosixPath('/content/drive/MyDrive/LLMBot/DataSet/ncert/class_10/Mathematics/jemh106.pdf'),
 PosixPath('/content/drive/MyDrive/LLMBot/DataSet/ncert/class_10/Mathematics/jemh107.pdf'),
 PosixPath('/content/drive/MyDrive/LLMBot/DataSet/ncert/class_10/Mathematics/jemh108.pdf'),
 PosixPath('/content/drive/MyDrive/LLMBot/DataSet/ncert/class_10/Mathematics/jemh109.pdf'),
 PosixPath('/content/drive/MyDrive/LLMBot/DataSet/ncert/class_10/Mathematics/jemh110.pdf')]

Every extracted page remains traceable to its PDF path and is checked for an empty text layer before it enters later stages.

### Extract, clean, and index pages

In [8]:
# Match textbook chapter headings like 'Chapter 1: Rational Numbers' or 'Lesson 2'.
CHAPTER_RE = re.compile(
    r"(?im)^\s*(?:(?:chapter|lesson)\s+\d+|chapter\s+[ivxlcdm]+)\s*[:.\-]?\s*.*$"
)

# Clean page text before storing it: remove PDF/control characters, fix broken hyphenation,
# and normalize whitespace so the later retrieval and model steps work with cleaner content.
def clean_page_text(text: str) -> str:
    text = text.replace("\x00", "").replace("\u00ad", "")
    text = re.sub(r"(?<=\w)-[ \t]*\n[ \t]*(?=\w)", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

# Keep a chapter label for each page. If a page does not contain a heading, reuse the most recent chapter.
def infer_chapter(text: str, previous: str = "Unknown") -> str:
    for line in text.splitlines():
        line = line.strip()
        if CHAPTER_RE.match(line) and len(line) < 160:
            return line
    return previous

# Create a list of extracted page records; each item is a unit of textbook content with context metadata.
page_records = []
empty_pages = []

# Read every PDF and process each page individually so the text can be linked back to its class, subject, chapter, and page number.
for pdf_path in pdf_paths:
    relative_parts = pdf_path.relative_to(DATA_ROOT).parts
    class_folder = next((part for part in relative_parts if part in SUPPORTED_GRADES), None)
    if class_folder is None:
        print(f"Skipping PDF outside class_6..class_10: {pdf_path}")
        continue
    class_number = SUPPORTED_GRADES[class_folder]
    class_index = relative_parts.index(class_folder)
    subject = relative_parts[class_index + 1] if class_index + 1 < len(relative_parts) - 1 else "Unknown"
    chapter = "Unknown"

    try:
        reader = PdfReader(str(pdf_path))
        for page_number, page in enumerate(reader.pages, start=1):
            raw_text = page.extract_text() or ""
            text = clean_page_text(raw_text)
            if not text:
                # Save empty pages for reporting; these may require OCR or a different extraction process.
                empty_pages.append({"source": str(pdf_path), "page": page_number})
                continue
            chapter = infer_chapter(text, chapter)
            page_records.append({
                "text": text,
                "class": class_number,
                "subject": subject,
                "chapter": chapter,
                "page": page_number,
                "source": str(pdf_path.resolve()),
                "document_id": str(pdf_path.relative_to(DATA_ROOT).with_suffix("")),
            })
    except Exception as exc:
        print(f"Could not process {pdf_path}: {exc}")

# Convert the cleaned page records into a DataFrame for summary checks and downstream chunking.
pages_df = pd.DataFrame(page_records)
if pages_df.empty:
    raise ValueError(f"No extractable text found in PDFs under {DATA_ROOT}")

print(f"Extracted {len(pages_df)} text pages; {len(empty_pages)} pages had no text (OCR may be needed).")
pages_df.groupby(["class", "subject"], as_index=False).agg(pages=("text", "size"))

Could not process /content/drive/MyDrive/LLMBot/DataSet/ncert/class_10/Science/jesc104.pdf: Reached maximum number of tokens per stream.
Could not process /content/drive/MyDrive/LLMBot/DataSet/ncert/class_10/Science/jesc105.pdf: Reached maximum number of tokens per stream.
Could not process /content/drive/MyDrive/LLMBot/DataSet/ncert/class_10/Science/jesc106.pdf: Reached maximum number of tokens per stream.
Could not process /content/drive/MyDrive/LLMBot/DataSet/ncert/class_10/Science/jesc108.pdf: Limit reached while decompressing. 244593 bytes remaining.
Extracted 3285 text pages; 0 pages had no text (OCR may be needed).


,class,subject,pages
0,6,Mathematics,412
1,6,Science,272
2,7,Mathematics,412
3,7,Science,208
4,8,Mathematics,455
5,8,Science,248
6,9,Mathematics,404
7,9,Science,412
8,10,Mathematics,276
9,10,Science,186


In [9]:
pages_df[["class", "subject", "chapter", "page", "text"]].head(3).assign(
    text=lambda frame: frame["text"].str.slice(0, 240)
)

,class,subject,chapter,page,text
0,10,Mathematics,Unknown,1,REAL NUMBERS 1\n1\n1.1 Introduction\nIn Class ...
1,10,Mathematics,Unknown,2,2 MATHEMATICS\n1.2 The Fundamental Theorem of ...
2,10,Mathematics,Unknown,3,REAL NUMBERS 3\nCarl Friedrich Gauss\n(1777 – ...


### Chapter-aware text chunking

Page records are split into overlapping chunks while preserving their metadata. Chunk size and overlap are explicit, reproducible settings; chapter headings are retained in the chunk context when present.

In [10]:
# Divide each cleaned page into overlapping text chunks so the model sees relevant context
# without exceeding token limits, while retaining page metadata for later retrieval and citation.
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 200
chunks = []

for page in page_records:
    text = page["text"]
    start = 0
    while start < len(text):
        end = min(start + CHUNK_SIZE, len(text))
        chunk_text = text[start:end].strip()
        if len(chunk_text) >= 80:
            # Store the same metadata with each chunk so source information remains available during training and RAG.
            chunks.append({**page, "text": chunk_text, "chunk_id": len(chunks)})
        if end == len(text):
            break
        start = end - CHUNK_OVERLAP

# Create a DataFrame of all chunks so they can be sampled, filtered, and saved for model training.
chunks_df = pd.DataFrame(chunks)
if chunks_df.empty:
    raise ValueError("No usable text chunks were created")
print(f"Created {len(chunks_df)} chunks from {len(page_records)} pages")
chunks_df[["class", "subject", "chapter", "page", "chunk_id", "text"]].head(2)

Created 5612 chunks from 3285 pages


,class,subject,chapter,page,chunk_id,text
0,10,Mathematics,Unknown,1,0,REAL NUMBERS 1\n1\n1.1 Introduction\nIn Class ...
1,10,Mathematics,Unknown,1,1,lication of positive integers. You already kno...


## 3. Dataset Creation

Generate short, source-grounded question-answer examples from textbook chunks, format them as instruction/chat records, then split by source document so pages from one book cannot leak across train, validation, and test.

In [11]:
# Load the base Phi-3 Mini model that will generate synthetic Q&A examples and later be fine-tuned to answer NCERT questions.
BASE_MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"
MAX_NEW_TOKENS = 96

# A CUDA GPU is required for this stage because the base model and quantized fine-tuning both need large memory and compute.
if not torch.cuda.is_available():
    raise RuntimeError(
        "Load Phi-3 Mini in a CUDA runtime for Q&A generation and QLoRA. "
        "Set the Colab runtime type to a T4 GPU and restart the runtime."
    )

# Use 4-bit quantization to reduce memory usage while still keeping the model usable for generation and fine-tuning.
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Initialize the tokenizer so prompts are formatted in the proper chat-template style expected by Phi-3.
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Load the quantized causal language model and keep it in evaluation mode until training begins.
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print(f"Successfully loaded {BASE_MODEL_ID} on {torch.cuda.get_device_name(0)}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Successfully loaded microsoft/Phi-3-mini-4k-instruct on Tesla T4


In [12]:
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Model placement:", getattr(model, "hf_device_map", "not available"))

CUDA: True
GPU: Tesla T4
Model placement: {'': 0}


In [13]:
# Verify and define OUTPUT_ROOT directory if not already set
from pathlib import Path

# Keep all generated artifacts in a dedicated folder so later checkpoints, models, and metrics remain organized.
try:
    # Check if OUTPUT_ROOT is already defined
    print(f"Target output directory: {OUTPUT_ROOT.resolve()}")
except NameError:
    PROJECT_ROOT = Path.cwd()
    OUTPUT_ROOT = PROJECT_ROOT / "artifacts"
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Created and initialized output directory: {OUTPUT_ROOT.resolve()}")

Target output directory: /content/artifacts


In [14]:
# This is the synthetic data creation stage: textbook chunks are converted into question-answer examples
# that will later train the model to answer in a curriculum-aware style.
import re
import json
import pandas as pd
import torch

MAX_NEW_TOKENS = 96
SEED = 42

# Create a single text completion from a prompt using the loaded Phi-3 model and tokenizer.
def generate_text(messages, max_new_tokens=MAX_NEW_TOKENS):
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    # Phi-3's custom generation path is not compatible with newer DynamicCache implementations.
    # Disabling cache avoids the `seen_tokens` AttributeError while keeping generation stable.
    with torch.inference_mode():
        output_ids = model.generate(
            inputs["input_ids"],
            attention_mask=inputs.get("attention_mask"),
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=False,
        )
    return tokenizer.decode(
        output_ids[0, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

# For each textbook chunk, ask the model to produce one valid Q&A pair in JSON format.
def generate_qa(chunk):
    prompt = (
        "Create one clear school-level question and its concise answer using only this NCERT passage. "
        "Return exactly one JSON object with keys question and answer. Do not include markdown.\n\n"
        f"Grade: {chunk['class']}\nSubject: {chunk['subject']}\nChapter: {chunk['chapter']}\n"
        f"Passage:\n{chunk['text']}"
    )
    raw = generate_text([{"role": "user", "content": prompt}], max_new_tokens=120)
    match = re.search(r"\{.*\}", raw, flags=re.DOTALL)
    if not match:
        return None
    try:
        qa = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None
    question, answer = qa.get("question", "").strip(), qa.get("answer", "").strip()
    if not question or not answer:
        return None
    return {
        "instruction": "Answer the question using only the supplied NCERT passage. If it does not contain the answer, say so.",
        "input": f"Grade {chunk['class']} | {chunk['subject']} | {chunk['chapter']}\n{chunk['text']}",
        "output": answer,
        "question": question,
        "class": chunk["class"],
        "subject": chunk["subject"],
        "chapter": chunk["chapter"],
        "source": chunk["source"],
        "document_id": chunk["document_id"],
        "page": chunk["page"],
        "chunk_id": int(chunk["chunk_id"]),
    }

# Use a moderate generation limit so the process stays feasible on the available GPU and still covers the dataset.
QA_SAMPLE_LIMIT = 25
CHECKPOINT_EVERY = 5
QA_CHECKPOINT_PATH = OUTPUT_ROOT / "ncert_qa_checkpoint.jsonl"

# Sample a balanced subset from each class/subject group so each area contributes to training without overloading a single group.
grouped_chunks = chunks_df.groupby(["class", "subject"], sort=True)
samples_per_group = max(1, (QA_SAMPLE_LIMIT + grouped_chunks.ngroups - 1) // grouped_chunks.ngroups)
qa_candidates = pd.concat([
    group.sample(n=min(len(group), samples_per_group), random_state=SEED)
    for _, group in grouped_chunks
]).head(QA_SAMPLE_LIMIT).reset_index(drop=True)

# Resume from a checkpoint if the notebook stops mid-generation, preventing repeated work on already processed chunks.
qa_records = []
if QA_CHECKPOINT_PATH.exists() and QA_CHECKPOINT_PATH.stat().st_size:
    try:
        checkpoint_df = pd.read_json(QA_CHECKPOINT_PATH, lines=True)
        if "chunk_id" in checkpoint_df.columns:
            candidate_ids = set(qa_candidates["chunk_id"].astype(int))
            checkpoint_df = checkpoint_df[checkpoint_df["chunk_id"].astype(int).isin(candidate_ids)]
            qa_records = checkpoint_df.to_dict(orient="records")
    except (ValueError, TypeError):
        qa_records = []

completed_chunk_ids = {int(record["chunk_id"]) for record in qa_records}
print(f"Generating Q&As for {len(qa_candidates)} chunks; {len(completed_chunk_ids)} already checkpointed.")

# Loop through candidate chunks, generate a Q&A record, and save progress every few examples.
for attempt, (_, chunk) in enumerate(qa_candidates.iterrows(), start=1):
    chunk_id = int(chunk["chunk_id"])
    if chunk_id in completed_chunk_ids:
        continue
    record = generate_qa(chunk)
    if record:
        qa_records.append(record)
        completed_chunk_ids.add(chunk_id)

    if attempt % CHECKPOINT_EVERY == 0 or attempt == len(qa_candidates):
        if qa_records:
            pd.DataFrame(qa_records).to_json(
                QA_CHECKPOINT_PATH,
                orient="records",
                lines=True,
                force_ascii=False,
            )
        print(f"Processed {attempt}/{len(qa_candidates)} chunks; saved {len(qa_records)} valid Q&As.")

# Build the final Q&A dataset and save it for training and evaluation.
qa_df = pd.DataFrame(qa_records)
if len(qa_df) < 3:
    raise ValueError("Too few valid generated Q&A records; review the model output before increasing QA_SAMPLE_LIMIT.")

qa_df.to_json(OUTPUT_ROOT / "ncert_qa.jsonl", orient="records", lines=True, force_ascii=False)
print(f"Generated {len(qa_df)} grounded Q&A records")
display(qa_df[["class", "subject", "chapter", "question", "output"]].head())

Generating Q&As for 25 chunks; 0 already checkpointed.


Processed 5/25 chunks; saved 5 valid Q&As.
Processed 10/25 chunks; saved 9 valid Q&As.
Processed 15/25 chunks; saved 14 valid Q&As.
Processed 20/25 chunks; saved 18 valid Q&As.
Processed 25/25 chunks; saved 22 valid Q&As.
Generated 22 grounded Q&A records


,class,subject,chapter,question,output
0,6,Mathematics,Chapter 4_Data Handling and Presentation.indd ...,On which day were the minimum number of books ...,Wednesday
1,6,Mathematics,CHAPTER 3 — SOLUTIONS,Can you fill a supercell table without repeati...,No; the ce
2,6,Mathematics,Chapter 5_Prime Time.indd 128 12-12-2024 10:52:09,How can you determine if two numbers are co-pr...,Two numbers are co-prime if their prime factor...
3,6,Science,Chapter 12.indd 253 10/4/2024 3:11:56 PM,What does the title of the book suggest about ...,The title of the book suggests that the journe...
4,6,Science,Chapter 3.indd 40 10/4/2024 3:17:26 PM,Why do marathon runners drink glucose water du...,Marathon runners drink glucose water during an...


In [15]:
# Hold out a document-level test split so answers are evaluated on unseen books rather than pages from the same source.
group_count = qa_df["document_id"].nunique()
if group_count < 6:
    raise ValueError("At least six source PDFs with generated Q&A are needed for non-empty document-grouped train/validation/test splits.")

# Split by document ID so the same NCERT book cannot appear in both training and evaluation sets.
def grouped_split(frame, test_size, random_state):
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    left_idx, right_idx = next(splitter.split(frame, groups=frame["document_id"]))
    return frame.iloc[left_idx].copy(), frame.iloc[right_idx].copy()

train_df, holdout_df = grouped_split(qa_df, test_size=0.20, random_state=SEED)
validation_df, test_df = grouped_split(holdout_df, test_size=0.50, random_state=SEED + 1)

# Turn each Q&A record into the chat-style format used by the causal language model during fine-tuning.
for split_name, split_df in [("train", train_df), ("validation", validation_df), ("test", test_df)]:
    if split_df.empty:
        raise ValueError(f"The {split_name} split is empty; add more source PDFs or Q&A examples.")
    split_df = split_df.copy()
    split_df["text"] = split_df.apply(
        lambda row: tokenizer.apply_chat_template(
            [
                {"role": "user", "content": f"{row['instruction']}\n\n{row['input']}\n\nQuestion: {row['question']}"},
                {"role": "assistant", "content": row["output"]},
            ],
            tokenize=False,
        ),
        axis=1,
    )
    split_df.to_json(OUTPUT_ROOT / f"{split_name}.jsonl", orient="records", lines=True, force_ascii=False)
    if split_name == "train":
        train_df = split_df
    elif split_name == "validation":
        validation_df = split_df
    else:
        test_df = split_df

split_summary = pd.DataFrame([
    {"split": name, "examples": len(frame), "documents": frame["document_id"].nunique()}
    for name, frame in [("train", train_df), ("validation", validation_df), ("test", test_df)]
])
display(split_summary)
assert not (set(train_df.document_id) & set(validation_df.document_id))
assert not (set(train_df.document_id) & set(test_df.document_id))
assert not (set(validation_df.document_id) & set(test_df.document_id))

,split,examples,documents
0,train,17,17
1,validation,2,2
2,test,3,3


## 4. Base LLM: Phi-3 Mini

In [16]:
# Evaluate a fixed sample from the held-out test set to compare the base and adapted model behaviors.
EVAL_LIMIT = 20
eval_df = test_df.sample(n=min(EVAL_LIMIT, len(test_df)), random_state=SEED).copy().reset_index(drop=True)

def build_question_prompt(row, context=None):
    # Build a task-specific prompt for a class-aware educational answer without forcing textbook support.
    prompt = (
        f"You are an NCERT learning assistant for Class {row['class']} {row['subject']}. "
        "Answer the student's question clearly and at the appropriate level. "
        "Do not claim textbook support unless a passage is supplied.\n"
    )
    if context:
        prompt += f"\nTextbook context:\n{context}\n"
    prompt += f"\nQuestion: {row['question']}\nAnswer:"
    return prompt

base_predictions = []
for _, row in eval_df.iterrows():
    base_predictions.append(generate_text([{"role": "user", "content": build_question_prompt(row)}]))

eval_df["base_answer"] = base_predictions
print(f"Recorded base-model answers for {len(eval_df)} held-out questions")

Recorded base-model answers for 3 held-out questions


Chunks preserve their page and source metadata for instruction generation, document-level splitting, semantic retrieval, and answer citations.

## 5. QLoRA Fine-tuning

QLoRA adapts the 4-bit Phi-3 Mini base model with low-rank adapters, keeping GPU memory requirements suitable for a T4. The adapter is trained only on the training split and monitored on validation data; the test split remains untouched until final comparison.

In [17]:
# Prepare the 4-bit model for adapter training and reduce activation memory with checkpointing.
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()
model.config.use_cache = False

# Configure LoRA adapters on Phi-3 attention and feed-forward projection layers.
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["qkv_proj", "o_proj", "gate_up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Convert each prepared text split to a Hugging Face Dataset and tokenize it for causal language modeling.
def tokenize_split(frame):
    dataset = Dataset.from_pandas(frame[["text"]], preserve_index=False)
    return dataset.map(
        lambda batch: tokenizer(batch["text"], truncation=True, max_length=1024),
        batched=True,
        remove_columns=["text"],
    )

# Keep training settings small enough for a Colab T4 while evaluating and saving once per epoch.
training_args = TrainingArguments(
    output_dir=str(OUTPUT_ROOT / "phi3-qlora"),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    logging_steps=10,
    report_to="none",
    seed=SEED,
)

# Train adapters on the training split and use validation data for epoch-level evaluation.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenize_split(train_df),
    eval_dataset=tokenize_split(validation_df),
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    tokenizer=tokenizer,
)
trainer.train()

# Save the adapter and tokenizer separately; the original Phi-3 base model is still required to reload them.
ADAPTER_DIR = OUTPUT_ROOT / "phi3-qlora-adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
# Keep cache disabled after LoRA training as well; Phi-3 remote generation is incompatible with the newer DynamicCache API.
model.config.use_cache = False
model.eval()
print(f"Saved QLoRA adapter to {ADAPTER_DIR}")

trainable params: 25,165,824 || all params: 3,846,245,376 || trainable%: 0.6543


Map:   0%|          | 0/17 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

/tmp/ipykernel_498/461164048.py:47: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,No log,1.513523


Saved QLoRA adapter to /content/artifacts/phi3-qlora-adapter


Text chunks were successfully transformed into semantic vector representations. These embeddings capture the meaning of the text and enable similarity-based retrieval.

## 6. RAG: Embeddings and Vector Database

Text chunks are embedded with `all-MiniLM-L6-v2` and indexed in FAISS using normalized vectors and inner-product similarity. The matching chunk records retain grade, subject, chapter, page, and source for grounded responses and citations.

In [18]:
# Select the sentence-transformer used to represent textbook chunks and learner questions in the same vector space.
EMBEDDING_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_ID, device=DEVICE)

# Encode every chunk in batches; normalized vectors allow inner-product search to behave like cosine similarity.
chunk_texts = chunks_df["text"].tolist()
chunk_vectors = embedding_model.encode(
    chunk_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
).astype("float32")

# Build a flat inner-product index and add vectors in the same order as chunk_texts and chunk_metadata.
vector_index = faiss.IndexFlatIP(chunk_vectors.shape[1])
vector_index.add(chunk_vectors)
chunk_metadata = chunks_df.drop(columns=["text"]).to_dict(orient="records")
print(f"Indexed {vector_index.ntotal} NCERT chunks ({chunk_vectors.shape[1]} dimensions)")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/88 [00:00<?, ?it/s]

Indexed 5612 NCERT chunks (384 dimensions)


In [19]:
# Keep the FAISS index and matching chunk metadata together so retrieval can be restored without re-embedding all text.
VECTOR_DIR = OUTPUT_ROOT / "faiss_ncert"
VECTOR_DIR.mkdir(parents=True, exist_ok=True)
faiss.write_index(vector_index, str(VECTOR_DIR / "index.faiss"))
pd.DataFrame(chunk_metadata).to_json(
    VECTOR_DIR / "metadata.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)
print(f"Saved FAISS index and chunk metadata to {VECTOR_DIR}")

Saved FAISS index and chunk metadata to /content/artifacts/faiss_ncert


### Validate semantic retrieval

In [20]:
# Search the FAISS index for passages related to a question, then apply optional class/subject filters.
def retrieve_chunks(query, grade=None, subject=None, top_k=3):
    # Return early for an invalid result count or an index that has no textbook chunks.
    if top_k <= 0 or vector_index.ntotal == 0:
        return []

    # Encode the query with the same model and normalization used when the textbook chunks were indexed.
    query_vector = embedding_model.encode(
        [query], normalize_embeddings=True, convert_to_numpy=True
    ).astype("float32")

    # Search all vectors so filtering by grade and subject does not hide eligible lower-ranked results.
    scores, indices = vector_index.search(query_vector, vector_index.ntotal)
    results = []
    for score, index_position in zip(scores[0], indices[0]):
        if index_position < 0:
            continue
        metadata = chunk_metadata[index_position]
        if grade is not None and int(metadata["class"]) != int(grade):
            continue
        if subject is not None and str(metadata["subject"]).casefold() != str(subject).casefold():
            continue
        # Keep the similarity score and source metadata alongside the passage for answer citations.
        results.append({**metadata, "text": chunk_texts[index_position], "score": float(score)})
        if len(results) == top_k:
            break
    return results

# Inspect a few matches to check their grade, subject, source location, score, and retrieved text.
retrieval_examples = retrieve_chunks("fractions and equivalent fractions", grade=6, top_k=3)
for item in retrieval_examples:
    print(f"Class {item['class']} | {item['subject']} | {item['chapter']} | p. {item['page']} | {item['score']:.3f}")
    print(item["text"][:350], "\n")

Class 6 | Mathematics | Chapter 7_Fractions.indd 186 13-08-2024 15:39:49 | p. 36 | 0.626
Ganita Prakash | Grade 6
186
  Fraction as equal share: When a whole number of units is divided 
into equal parts and shared equally, a fraction results. 
  Fractional Units: When one whole basic unit is divided into equal 
parts, then each part is called a fractional unit. 
  Reading Fractions: In a fraction such as 56, 5 is called the numerato 

Class 6 | Mathematics | Chapter 7_Fractions.indd 172 13-08-2024 15:39:46 | p. 22 | 0.554
Ganita Prakash | Grade 6
172
 Find equivalent fractions for the given pairs of fractions such that the 
fractional units are the same.
a. 72 and 35 b. 83 and 56 c. 34 and 35 d. 67 and 85 
e. 94 and 52 f. 110 and 29 g. 83 and 114 h. 136 and 19 
Expressing a fraction in lowest terms (or in its simplest 
form) 
In any fraction, if its numerator and de 

Class 6 | Mathematics | Chapter 7_Fractions.indd 181 13-08-2024 15:39:48 | p. 31 | 0.550
Fractions
181
 Figure it Ou

In [21]:
# Smoke-test retrieval with representative class/subject questions to check filtering and inspect ranked textbook passages.
sample_queries = [
    (6, "Mathematics", "What are equivalent fractions?"),
    (7, "Science", "How do plants transport water?"),
    (8, "Mathematics", "How do linear equations work?"),
]
for grade, subject, query in sample_queries:
    results = retrieve_chunks(query, grade=grade, subject=subject, top_k=2)
    print(f"\nClass {grade} {subject}: {query}")
    if not results:
        print("No matching textbook passage found.")
    for result in results:
        print(f"{result['chapter']} | page {result['page']} | score {result['score']:.3f}")
        print(result["text"][:500])


Class 6 Mathematics: What are equivalent fractions?
Chapter 7_Fractions.indd 186 13-08-2024 15:39:49 | page 36 | score 0.714
Ganita Prakash | Grade 6
186
  Fraction as equal share: When a whole number of units is divided 
into equal parts and shared equally, a fraction results. 
  Fractional Units: When one whole basic unit is divided into equal 
parts, then each part is called a fractional unit. 
  Reading Fractions: In a fraction such as 56, 5 is called the numerator 
and 6 is called the denominator .
  Mixed fractions contain a whole number part and a fractional part. 
  Number line: Fractions can be shown on 
Chapter 7_Fractions.indd 172 13-08-2024 15:39:46 | page 22 | score 0.576
Ganita Prakash | Grade 6
172
 Find equivalent fractions for the given pairs of fractions such that the 
fractional units are the same.
a. 72 and 35 b. 83 and 56 c. 34 and 35 d. 67 and 85 
e. 94 and 52 f. 110 and 29 g. 83 and 114 h. 136 and 19 
Expressing a fraction in lowest terms (or in its simples

### Context-grounded answer generation

In [22]:
# Retrieve passages that match the question and any class/subject filters selected by the learner.
def rag_answer(question, grade=None, subject=None, top_k=2):
    retrieved = retrieve_chunks(question, grade=grade, subject=subject, top_k=top_k)
    if not retrieved:
        return "I could not find a matching passage in the selected NCERT textbooks.", []

    # Include source labels in the prompt so the model can use the right grade-level context.
    context = "\n\n".join(
        f"[Class {item['class']} | {item['subject']} | {item['chapter']} | page {item['page']}]\n{item['text']}"
        for item in retrieved
    )

    # Explicitly constrain the response to retrieved evidence and require an honest fallback when evidence is missing.
    prompt = (
        "You are an NCERT learning assistant. Answer using only the textbook context below. "
        "If the context does not support an answer, say that the information was not found. "
        "Explain clearly for the selected class and do not invent facts.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )
    answer = generate_text([{"role": "user", "content": prompt}])

    # Return citation fields separately so the UI can show where the supporting passages came from.
    sources = [
        {"class": item["class"], "subject": item["subject"], "chapter": item["chapter"],
         "page": item["page"], "source": item["source"], "score": item["score"]}
        for item in retrieved
    ]
    return answer, sources

## 7. Gradio Chatbot

The Gradio interface accepts a question and lets learners constrain retrieval by class and subject. Each answer includes the retrieved chapter, page, and source path so learners can return to the textbook passage.

In [23]:
# Adapt the UI's class and subject selections to the optional filters used by textbook retrieval.
def format_chat_response(question, grade, subject):
    answer, sources = rag_answer(
        question,
        grade=int(grade) if grade else None,
        subject=subject if subject != "All subjects" else None,
    )

    # Append concise textbook citations so learners can verify the passages behind the response.
    if sources:
        citations = "\n\nSources:\n" + "\n".join(
            f"Class {item['class']} {item['subject']} | {item['chapter']} | page {item['page']} | {Path(item['source']).name}"
            for item in sources
        )
        answer += citations
    return answer

In [24]:
import gradio as gr

# Populate the dropdowns from the supported class range and subjects actually found in the processed PDFs.
class_choices = [str(grade) for grade in range(6, 11)]
subject_choices = ["All subjects"] + sorted(pages_df["subject"].dropna().astype(str).unique().tolist())

# Connect the response formatter to a simple question form with class and subject retrieval filters.
interface = gr.Interface(
    fn=format_chat_response,
    inputs=[
        gr.Textbox(label="Question", lines=3, placeholder="Ask a question about an NCERT topic"),
        gr.Dropdown(choices=class_choices, value="6", label="Class"),
        gr.Dropdown(choices=subject_choices, value="All subjects", label="Subject"),
    ],
    outputs=gr.Textbox(label="Textbook-grounded answer and sources", lines=10),
    title="NCERT Learning Assistant",
    description="Ask a question and receive an answer grounded in the selected NCERT textbooks.",
)
print("Gradio interface configured. Launch it in the next cell.")

Gradio interface configured. Launch it in the next cell.


In [25]:
def launch_chatbot():
    """Start the Gradio app only after the model and retrieval assets are ready."""
    interface.launch(share=True, prevent_thread_lock=True)

# Run the chatbot separately from the training and evaluation workflow.
launch_chatbot()

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2cb9568d297bd71fa0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [28]:
import re
import pandas as pd
import numpy as np

def token_f1(reference, prediction):
    reference_tokens = re.findall(r"\w+", str(reference).casefold())
    prediction_tokens = re.findall(r"\w+", str(prediction).casefold())
    if not reference_tokens or not prediction_tokens:
        return 0.0
    reference_counts = pd.Series(reference_tokens).value_counts().to_dict()
    prediction_counts = pd.Series(prediction_tokens).value_counts().to_dict()
    overlap = sum(
        min(reference_counts[token], prediction_counts[token])
        for token in reference_counts.keys() & prediction_counts.keys()
    )
    if overlap == 0:
        return 0.0
    precision = overlap / len(prediction_tokens)
    recall = overlap / len(reference_tokens)
    return 2 * precision * recall / (precision + recall)

def run_evaluation():
    """Run model comparison independently from the chatbot startup path."""
    if not hasattr(model, "peft_config"):
        raise RuntimeError("Run the QLoRA training cell before evaluating the fine-tuned model.")

    qlora_predictions = []
    for _, row in eval_df.iterrows():
        qlora_predictions.append(generate_text([{"role": "user", "content": build_question_prompt(row)}]))
    eval_df["qlora_answer"] = qlora_predictions

    base_rag_predictions = []
    with model.disable_adapter():
        for _, row in eval_df.iterrows():
            answer, _ = rag_answer(row["question"], grade=row["class"], subject=row["subject"])
            base_rag_predictions.append(answer)
    eval_df["base_rag_answer"] = base_rag_predictions

    qlora_rag_predictions = []
    for _, row in eval_df.iterrows():
        answer, _ = rag_answer(row["question"], grade=row["class"], subject=row["subject"])
        qlora_rag_predictions.append(answer)
    eval_df["qlora_rag_answer"] = qlora_rag_predictions

    eval_df[["class", "subject", "question", "base_answer", "base_rag_answer", "qlora_answer", "qlora_rag_answer"]].head()

    prediction_columns = {
        "base": "base_answer",
        "base_rag": "base_rag_answer",
        "qlora": "qlora_answer",
        "qlora_rag": "qlora_rag_answer",
    }

    reference_column = "output"
    metric_rows = []
    for mode, column in prediction_columns.items():
        scores = [
            token_f1(reference, prediction)
            for reference, prediction in zip(eval_df[reference_column], eval_df[column])
        ]
        metric_rows.append({
            "mode": mode,
            "mean_token_f1": float(np.mean(scores)),
            "examples": len(scores),
        })

    evaluation_results = pd.DataFrame(metric_rows)
    evaluation_results.to_csv(OUTPUT_ROOT / "evaluation_results.csv", index=False)
    return evaluation_results

# Run evaluation only when you explicitly want to compare model variants.
evaluation_results = run_evaluation()
display(evaluation_results)

,mode,mean_token_f1,examples
0,base,0.203186,3
1,base_rag,0.217178,3
2,qlora,0.194885,3
3,qlora_rag,0.246694,3


## 8. Evaluation

In [30]:
print("All four inference modes are compared on the same document-held-out test set.")
print("Review the generated references and model outputs before interpreting token F1 as educational correctness.")

All four inference modes are compared on the same document-held-out test set.
Review the generated references and model outputs before interpreting token F1 as educational correctness.


## Results



The outcome is a working educational chatbot that demonstrates how fine-tuning and retrieval can be combined to support school-level learning. The project  produce a prepared NCERT knowledge base, a fine-tuning dataset, an adapted language model, an interactive application, and a comparative evaluation of the proposed approach.

## **Challenges Faced**

Some of the challenges faced during the project are:

1. Library compatibility
2. Slow text generation
3. Limited GPU memory

### Limitations

Text-only extraction may not preserve all diagrams, tables, or equations. Fine-tuning on generated examples may introduce errors if the examples are not validated. Retrieval also cannot guarantee correctness if the relevant passage is not indexed or is ranked poorly.

The initial implementation is limited to the books and subjects actually processed. Results from a limited pilot should not be generalized to every NCERT class or subject.

The chatbot is an educational support tool and should not be presented as a replacement for teachers, textbooks, or independent verification of important answers.



## Future Enhancements

1. Add and verify NCERT PDFs for Social Science and English for class 6 to 10 under their matching class/subject folders.
2. May be extend up to class 12th.
3. Supporting diagrams and image-based textbook questions.
4. Supporting personalized quizzes and revision plans.
5. Integration of larger and more capable language models.
6. Addition of multilingual support for regional languages.
7. Voice-based interaction using speech recognition.
8. Incorporation of conversational memory for multi-turn discussions.
9. Deployment as a web application or mobile application.

## Conclusion

This project successfully developed  NCERT assistant workflow from textbook collection through PDF processing, dataset creation, Phi-3 Mini, optional QLoRA on a T4, RAG, Gradio, and four-way evaluation.

This project applies modern generative AI techniques to the education domain to make textbook-based learning more interactive and accessible. By combining parameter-efficient LLM fine-tuning with retrieval from NCERT textbooks, the system aims to deliver understandable, curriculum-aligned answers while reducing unsupported responses. The initial implementation will focus on Classes 6–10 Science and Mathematics, with the architecture designed to support future expansion to higher classes and additional subjects.

End of NCERT Learning Assistant pipeline